# Tema 3 - Laborator 4

In [14]:
# Importuri
import numpy as np
from scipy.stats import binom

from pgmpy.models import DiscreteBayesianNetwork   # clasa pentru rețeaua Bayesiană
from pgmpy.factors.discrete import TabularCPD      # clasa pentru tabelele de probabilități
from pgmpy.inference import VariableElimination    # algoritmul care face inferența

---
# Ex 1

## Enuntul problemei
La un concurs TV și ai 3 usi. În spatele uneia se afla o masina, iar în spatele celorlalte două sunt capre.

1. Alegi o usa (să zicem ușa 0).
2. Prezentatorul, care stie unde e masina, deschide o alta usa, în spatele căreia știe sigur că e o capră (să zicem ușa 2).
3. Te întreaba: „Vrei să te razgandesti si să alegi usa 1?”

Merita să schimbi ușa?

Intuitia spune „e 50-50”. Dar răspunsul corect este că **da, merita sa schimbi**: schimband, ai $\tfrac{2}{3}$ sanse să castigi, iar ramanand, doar $\tfrac{1}{3}$.

## Cum transformăm problema în rețea Bayesiană
Avem 3 variabile, fiecare cu 3 valori posibile (ușa 0, 1 sau 2):

| Variabila | meaning |
|---|---|
| **C** (Contestant) | usa aleasa de concurent |
| **P** (Prize) | usa în spatele căreia e premiul (mașina) |
| **H** (Host) | ușa deschisa de prezentator |

Relațiile dintre ele:
- Concurentul alege ușa la întâmplare, deci $P(C=0)=P(C=1)=P(C=2)=\tfrac13$.
- Premiul e pus la întâmplare, deci $P(P=0)=P(P=1)=P(P=2)=\tfrac13$.
- Prezentatorul alege ușa în funcție de ce a ales concurentul si de unde e premiul (nu poate deschide nici usa concurentului, nici usa cu premiul). Deci $H$ are doi parinti: $C$ și $P$.

Graful este: $C \to H \leftarrow P$.

### Tabelul $P(H \mid C, P)$
Prezentatorul deschide mereu o usa care nu e aleasa de concurent și nu are premiul.
- Daca $C \ne P$ (concurentul a ales o capra), prezentatorul are o singura ușa posibila (cea ramasa), deci o deschide cu probabilitate 1.
- Daca $C = P$ (concurentul a nimerit mașina), prezentatorul poate deschide oricare din celelalte 2 usi, deci fiecare cu probabilitate $\tfrac12$.

In [15]:
# 1) Structura retelei: C -> H și P -> H
model = DiscreteBayesianNetwork([("C", "H"), ("P", "H")])

# 2) Tabelele de probabilitati
# Prima ușă aleasa: 3 stări, toate la fel de probabile.
# Folosim 1/3
cpd_c = TabularCPD("C", 3, [[1/3], [1/3], [1/3]])

# Usa cu premiul: 3 stari, toate la fel de probabile.
cpd_p = TabularCPD("P", 3, [[1/3], [1/3], [1/3]])

# Ușa deschisa de prezentator, P(H | C, P).
# Fiecare rand = o valoare a lui H (H=0, H=1, H=2).
# Fiecare Ccoloana = o combinatie (C, P), în ordinea:
#   (C=0,P=0) (C=0,P=1) (C=0,P=2) (C=1,P=0) (C=1,P=1) (C=1,P=2) (C=2,P=0) (C=2,P=1) (C=2,P=2)
cpd_h = TabularCPD(
    "H",
    3,
    [
        [0,   0, 0,   0, 0.5, 1,   0, 1, 0.5],   # H= 0
        [0.5, 0, 1,   0, 0,   0,   1, 0, 0.5],   # H=1
        [0.5, 1, 0,   1, 0.5, 0,   0, 0, 0  ],   # H=2
    ],
    evidence=["C", "P"], # parintii lui H
    evidence_card=[3, 3],# fiecare parinte are 3 stari
)

# 3) Atasam tabelele la rețea
model.add_cpds(cpd_c, cpd_p, cpd_h)

# 4) Verificăm că reteaua e corecta (structură + tabele care însumează 1)
print("Modelul este valid:", model.check_model())

Modelul este valid: True


**Exemplu de citire a tabelului:** coloana $(C=0, P=1)$ = „concurentul a ales usa 0, masina e în spatele usii 1”. Prezentatorul nu poate deschide usa 0 (a ales-o concurentul) și nici ușa 1 (are masina), deci deschide **sigur** usa 2. În tabel, pe coloana asta avem $P(H=2)=1$ și 0 în rest.

## Inferența
Sa presupunem ca concurentul a ales ușa 0 și prezentatorul a deschis ușa 2. Vrem $P(P \mid C=0, H=2)$, adica „unde e probabil mașina, stiind ce am observat?”.

In [16]:
# Cream obiectul care face inferența
infer = VariableElimination(model)

# are e distributia lui P, dacă C=0 și H=2?
posterior_p = infer.query(["P"], evidence={"C": 0, "H": 2}, show_progress=False)
print(posterior_p)

+------+----------+
| P    |   phi(P) |
+======+==========+
| P(0) |   0.3333 |
+------+----------+
| P(1) |   0.6667 |
+------+----------+
| P(2) |   0.0000 |
+------+----------+


**Interpretare:**
- $P(P=0 \mid \dots) = \tfrac13$: daca ramai la usa 0, ai doar $\tfrac13$ sanse.
- $P(P=1 \mid \dots) = \tfrac23$: daca schimbi pe usa 1, ai $\tfrac23$ sanse.
- $P(P=2 \mid \dots) = 0$: usa 2 a fost deschisa si avea capra.

### Verificare cu regula lui Bayes
$$P(P=p \mid C=0,H=2) \;\propto\; P(P=p)\cdot P(C=0)\cdot P(H=2 \mid C=0,P=p)$$
Factorii $P(C=0)=\tfrac13$ sunt la fel pentru toti $p$, deci conteaza doar $P(P=p)\cdot P(H=2\mid C=0,P=p)$:
- $p=0$: $\tfrac13\cdot\tfrac12=\tfrac16$
- $p=1$: $\tfrac13\cdot 1=\tfrac13$
- $p=2$: $\tfrac13\cdot 0=0$

Suma = $\tfrac12$. Normalizam: $\tfrac{1/6}{1/2}=\tfrac13$, $\tfrac{1/3}{1/2}=\tfrac23$, $0$.


---
# Ex 2

Variabile:
- **B** - pacientul are boala
- **T** - tuse
- **D** - dificultati de respiratie
- **X** - radiografie anormala

Din enunț:
- boala (B) influenteaza tusea (T) $\Rightarrow B \to T$
- boala (B) influențează radiografia (X) $\Rightarrow B \to X$
- dificultatea de respiratie (D) depinde de boala si de tuse $\Rightarrow B \to D$ și $T \to D$

## a) Definirea modelului

In [17]:
# Structura rețelei: B -> T, B -> X, B -> D, T -> D
model2 = DiscreteBayesianNetwork([("B", "T"), ("B", "X"), ("B", "D"), ("T", "D")])

# P(B): randul 0= P(B=0), randul 1=P(B=1)
cpd_B = TabularCPD("B", 2, [[0.9], [0.1]])

# P(T | B): coloanele sunt B=0, B=1
#   P(T=1|B=0)=0.3 ; P(T=1|B=1)=0.8
cpd_T = TabularCPD(
    "T", 2,
    [[0.7, 0.2],  # T = 0
     [0.3, 0.8]], # T = 1
    evidence=["B"], evidence_card=[2],
)

# P(X | B): coloanele sunt B=0, B=1
#   P(X=1|B=0)=0.1 ; P(X=1|B=1)=0.9
cpd_X = TabularCPD(
    "X", 2,
    [[0.9, 0.1], # X = 0
     [0.1, 0.9]],# X = 1
    evidence=["B"], evidence_card=[2],
)

# P(D | B, T): coloanele sunt, (B=0,T=0) (B=0,T=1) (B=1,T=0) (B=1,T=1)
#   P(D=1|B=0,T=0)=0.1 ; P(D=1|B=0,T=1)=0.5 ; P(D=1|B=1,T=0)=0.4 ; P(D=1|B=1,T=1)=0.9
cpd_D = TabularCPD(
    "D", 2,
    [[0.9, 0.5, 0.6, 0.1],   # D = 0
     [0.1, 0.5, 0.4, 0.9]],  # D = 1
    evidence=["B", "T"], evidence_card=[2, 2],
)

model2.add_cpds(cpd_B, cpd_T, cpd_X, cpd_D)
print("Modelul este valid:", model2.check_model())

# Afișam tabelele 
for cpd in model2.get_cpds():
    print(cpd)
    print()

Modelul este valid: True
+------+-----+
| B(0) | 0.9 |
+------+-----+
| B(1) | 0.1 |
+------+-----+

+------+------+------+
| B    | B(0) | B(1) |
+------+------+------+
| T(0) | 0.7  | 0.2  |
+------+------+------+
| T(1) | 0.3  | 0.8  |
+------+------+------+

+------+------+------+
| B    | B(0) | B(1) |
+------+------+------+
| X(0) | 0.9  | 0.1  |
+------+------+------+
| X(1) | 0.1  | 0.9  |
+------+------+------+

+------+------+------+------+------+
| B    | B(0) | B(0) | B(1) | B(1) |
+------+------+------+------+------+
| T    | T(0) | T(1) | T(0) | T(1) |
+------+------+------+------+------+
| D(0) | 0.9  | 0.5  | 0.6  | 0.1  |
+------+------+------+------+------+
| D(1) | 0.1  | 0.5  | 0.4  | 0.9  |
+------+------+------+------+------+



## b) Pacientul are tuse și dificultati de respirație. Care e probabilitatea să aiba boala?
Căutăm $P(B=1 \mid T=1, D=1)$.

In [18]:
infer2 = VariableElimination(model2)

rez_b = infer2.query(["B"], evidence={"T": 1, "D": 1}, show_progress=False)
print(rez_b)

+------+----------+
| B    |   phi(B) |
+======+==========+
| B(0) |   0.6522 |
+------+----------+
| B(1) |   0.3478 |
+------+----------+


## c) Se află că pacientul nu are boala. Care e probabilitatea ca radiografia sa fi fost anormala?
Cautam $P(X=1 \mid B=0)$.

In [19]:
rez_c = infer2.query(["X"], evidence={"B": 0}, show_progress=False)
print(rez_c)

+------+----------+
| X    |   phi(X) |
+======+==========+
| X(0) |   0.9000 |
+------+----------+
| X(1) |   0.1000 |
+------+----------+


## d) Calculul cu regula lui Bayes


Distribuția comună se obține înmultind tabelele locale, fiecare nod conditionat de părinții lui:
$$P(B,T,X,D)=P(B)\,P(T\mid B)\,P(X\mid B)\,P(D\mid B,T)$$

### Independențe condiționale
Citim din graf
- Dacă știm $B$, variabila $T$ nu mai depinde de nimic altceva în afara de $B$.
- $X$ are ca singur parinte pe $B$. Deci, daca știm $B$, $X$ este independenta de $T$ și $D$: $\;X \perp (T, D) \mid B$.

### Partea b)
Regula lui Bayes:
$$P(B=1 \mid T=1, D=1)=\frac{P(T=1, D=1 \mid B=1)\,P(B=1)}{P(T=1, D=1)}$$

**Numărătorul.** Din lanțul de probabilități (regula produsului):
$$P(T=1, D=1 \mid B=1)=P(T=1\mid B=1)\cdot P(D=1\mid B=1, T=1)=0.8\cdot 0.9=0.72$$
Deci numărătorul este $0.72 \cdot 0.1 = 0.072$.

**Numitorul** (formula probabilității totale, împărțind după cele două valori ale lui $B$):
$$P(T=1, D=1)=P(T=1,D=1\mid B=1)P(B=1)+P(T=1,D=1\mid B=0)P(B=0)$$
Calculăm al doilea termen:
$$P(T=1, D=1 \mid B=0)=P(T=1\mid B=0)\cdot P(D=1\mid B=0, T=1)=0.3\cdot 0.5=0.15$$
Atunci:
$$P(T=1, D=1)=0.72\cdot 0.1+0.15\cdot 0.9=0.072+0.135=0.207$$

**Rezultat:**
$$P(B=1 \mid T=1, D=1)=\frac{0.072}{0.207}=\frac{8}{23}\approx 0.3478$$

(Deci, chiar și cu simptomele astea, probabilitatea de boală e doar ~35%, pentru că boala e rara, apriori doar 10%.)

### Partea c)
Pentru ca $X \perp (T,D) \mid B$ și $B=0$ este cunoscut, nu mai conteaza altceva:
$$P(X=1 \mid B=0)=0.1$$
adică exact valoarea data direct în tabelul lui $X$.

---
# Ex 3

**Regulile jocului:**
1. Se arunca o monedă normala ca să se decida cine începe: $J_0$ sau $J_1$.
2. **Runda 1:** jucatorul care începe aruncă propriul zar, obține $n$ (între 1- 6).
3. **Runda 2:** celalalt jucător arunca **moneda lui** de $2n$ ori, iar $m$ = numărul de steme obținute.
4. Cine a început câștigă dacă $n \ge m$; altfel câștigă celălalt.

Mai știm:
- $J_0$ are monedă normala: $P(\text{stemă})=\tfrac12$.
- $J_1$ are moneda cu : $P(\text{stemă})=\tfrac47$.
- Zarurile sunt normale.

## a) Simulare: cine are șanse mai mari?

In [20]:
rng = np.random.default_rng(123)   # seed fix, ca rezultatele să fie repetabile

P_STEMA = {0: 1/2, 1: 4/7}# probabilitatea de stema pentru moneda lui J0, respectiv J1

def joaca_un_joc():
    # moneda care decide cine începe: 0 -> J0, 1 -> J1
    incepe = rng.integers(0, 2)
    al_doilea = 1 - incepe

    # runda 1: cel care incepe aruncă zarul (valori 1..6)
    n = rng.integers(1, 7)

    #  runda 2: celalalt arunca moneda lui de 2n ori si numaram stemele
    aruncari = rng.random(2 * n) < P_STEMA[al_doilea]   # True = stema
    m = aruncari.sum()

    
    if n >= m:
        castigator = incepe
    else:
        castigator = al_doilea
    return castigator

NR_JOCURI = 10_000
rezultate = [joaca_un_joc() for _ in range(NR_JOCURI)]

victorii_J0 = rezultate.count(0)
victorii_J1 = rezultate.count(1)
print(f"J0 a câștigat {victorii_J0} jocuri ({victorii_J0 / NR_JOCURI:.4f})")
print(f"J1 a câștigat {victorii_J1} jocuri ({victorii_J1 / NR_JOCURI:.4f})")

J0 a câștigat 4209 jocuri (0.4209)
J1 a câștigat 5791 jocuri (0.5791)


(Cine are mai multe victorii in simulare are șanse mai mari)

## b) Reteaua Bayesiana
Variabile:
- **S** (starter): cine incepe. $S=0$ înseamna $J_0$, $S=1$ înseamna $J_1$. Prior: $\tfrac12$ fiecare.
- **N**: nr de pe zar în prima runda, valori $1,\dots,6$, fiecare cu probabilitate $\tfrac16$.
- **M**: numărul de steme în runda a doua, valori $0,\dots,12$ (maxim $2\cdot 6 = 12$ aruncări).
  - $M$ depinde de **S** (pentru că ne spune *a cui* e moneda aruncată în runda 2) și de **N** (pentru ca ne spune *de câte ori* se aruncă).
  - Dacă $S=0$, în runda 2 arunca $J_1$: $M \sim \text{Binomial}(2n, \tfrac47)$.
  - Dacă $S=1$, în runda 2 aruncă $J_0$: $M \sim \text{Binomial}(2n, \tfrac12)$.
- **W** (winner): cine câștiga. Este determinat de $S$, $N$ și $M$ (regula jocului), deci tabelul lui conține doar 0 și 1.

Graful: $S \to M$, $N \to M$, $S \to W$, $N \to W$, $M \to W$.

In [21]:
model3 = DiscreteBayesianNetwork([("S", "M"), ("N", "M"), ("S", "W"), ("N", "W"), ("M", "W")])

valori_N = [1, 2, 3, 4, 5, 6]
valori_M = list(range(0, 13))     # 0..12

# P(S): cine începe (monedă normală)
cpd_S = TabularCPD("S", 2, [[0.5], [0.5]], state_names={"S": [0, 1]})

# P(N): zar normal
cpd_N = TabularCPD("N", 6, [[1/6]] * 6, state_names={"N": valori_N})

# P(M | S, N): construim tabelul cu un ciclu.
# Coloanele sunt în ordinea: pentru fiecare S (0, apoi 1), pentru fiecare N (1..6).
coloane_M = []
for s in [0, 1]:
    al_doilea = 1 - s   # cine aruncă în runda 2
    p = P_STEMA[al_doilea]    # probabilitatea de stemă a monedei lui
    for n in valori_N:
        # P(M=m | S=s, N=n) = Binomial(2n, p) evaluat în m (e 0 dacă m > 2n)
        coloana = [binom.pmf(m, 2 * n, p) for m in valori_M]
        coloane_M.append(coloana)

# transformăm lista de coloane într-o matrice cu 13 rânduri (m=0..12) și 12 coloane
tabel_M = np.array(coloane_M).T

cpd_M = TabularCPD(
    "M", 13, tabel_M,
    evidence=["S", "N"], evidence_card=[2, 6],
    state_names={"M": valori_M, "S": [0, 1], "N": valori_N},
)

# P(W | S, N, M): determinist. Coloanele: pentru fiecare S, apoi N, apoi M.
coloane_W = []
for s in [0, 1]:
    for n in valori_N:
        for m in valori_M:
            primul = s
            al_doilea = 1 - s
            castigator = primul if n >= m else al_doilea
            # coloana = [P(W=0), P(W=1)]
            coloane_W.append([1.0, 0.0] if castigator == 0 else [0.0, 1.0])

tabel_W = np.array(coloane_W).T       # formă (2, 156)

cpd_W = TabularCPD(
    "W", 2, tabel_W,
    evidence=["S", "N", "M"], evidence_card=[2, 6, 13],
    state_names={"W": [0, 1], "S": [0, 1], "N": valori_N, "M": valori_M},
)

model3.add_cpds(cpd_S, cpd_N, cpd_M, cpd_W)
print("Modelul este valid:", model3.check_model())
print("Muchiile rețelei:", list(model3.edges()))

Modelul este valid: True
Muchiile rețelei: [('S', 'M'), ('S', 'W'), ('M', 'W'), ('N', 'M'), ('N', 'W')]


## c) Cine a început jocul, știind că în runda a doua s-a obținut o singură stemă?
Căutăm $P(S \mid M=1)$ și alegem valoarea mai mare.

In [22]:
rez_S = infer3.query(["S"], evidence={"M": 1}, show_progress=False)
print(rez_S)

# Cine e cel mai probabil?
cel_mai_probabil = rez_S.state_names["S"][int(np.argmax(rez_S.values))]
print(f"\nCel mai probabil a început J{cel_mai_probabil}")

+------+----------+
| S    |   phi(S) |
+======+==========+
| S(0) |   0.4529 |
+------+----------+
| S(1) |   0.5471 |
+------+----------+

Cel mai probabil a început J1
